# Fase 13 — Analisis Data Benchmark Provider vs Riverpod

Alur: gabung JSON → frekuensi aktual → boxplot & *outlier* → deskriptif (CI 95%) →
Shapiro-Wilk per kelompok → Welch's t-test / Mann-Whitney U → Holm-Bonferroni (24 perbandingan) → *effect size*.

Sumber data dapat diganti lewat variabel lingkungan:
- `BENCH_RAW_DIR` (default `results/raw`; untuk pilot: `results/pilot/raw`)
- `BENCH_OUT_DIR` (default `results`)

In [ ]:
import json, os, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
RAW_DIR = Path(os.environ.get('BENCH_RAW_DIR', ROOT / 'results' / 'raw'))
OUT_DIR = Path(os.environ.get('BENCH_OUT_DIR', ROOT / 'results'))
TABLES, FIGURES = OUT_DIR / 'tables', OUT_DIR / 'figures'
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
CODES = ['A-N', 'A-S', 'B-N', 'B-S']
FREQS = [10, 20, 50, 90]
# Variabel terikat utama: kolom → (label, satuan)
METRICS = {
    'rebuilds_per_s_main': ('Rebuild count', 'widget/detik'),
    'frames.total_mean_ms': ('Frame rendering time (build + raster)', 'ms/frame'),
    'memory_mb.mean': ('Memory (rata-rata RSS)', 'MB'),
}
pd.set_option('display.width', 200, 'display.max_columns', 40)
print('RAW_DIR =', RAW_DIR)

## 1. Gabungkan semua JSON → `all_sessions.csv`

In [ ]:
records = []
for path in sorted(RAW_DIR.glob('*.json')):
    data = json.loads(path.read_text(encoding='utf-8'))
    m = re.search(r'_r(\d+)\.json$', path.name)
    data['run'] = int(m.group(1)) if m else None
    data['file'] = path.name
    records.append(data)

df = pd.json_normalize(records)
df = df.drop(columns=[c for c in df.columns if c.endswith('memory_mb.samples')])
df['hz'] = df['target_hz'].astype(int)
df = df.sort_values(['code', 'hz', 'run']).reset_index(drop=True)
df.to_csv(OUT_DIR / 'all_sessions.csv', index=False)

counts = df.pivot_table(index='code', columns='hz', values='file', aggfunc='count').reindex(index=CODES, columns=FREQS)
print(f'{len(df)} sesi dimuat')
counts

### Pemeriksaan kualitas data

In [ ]:
checks = pd.DataFrame({
    'build_mode_bukan_release': (df['build_mode'] != 'release'),
    'ada_tick_terlewat': (df['skipped_ticks'] > 0),
    'frame_window_cadangan': (df['frames.window_method'] != 'frame_number'),
    'frame_hilang': (df['frames.expected_count'] > 0) & (df['frames.count'] < df['frames.expected_count']),
})
print('Jumlah sesi yang ditandai per pemeriksaan:')
print(checks.sum().to_string())
print('\nRefresh rate perangkat:', sorted(df['device_refresh_hz'].unique()))
df.loc[checks.any(axis=1), ['file'] + list(checks.columns[checks.loc[checks.any(axis=1)].any()])]

## 2. Frekuensi aktual vs target

In [ ]:
freq = (df.groupby(['code', 'hz'])['actual_hz']
          .agg(['mean', 'std', 'min', 'max']).round(3)
          .assign(selisih_pct=lambda t: ((t['mean'] / t.index.get_level_values('hz') - 1) * 100).round(2)))
freq.to_csv(TABLES / 'frekuensi_aktual.csv')
freq

## 3. Boxplot per konfigurasi dan identifikasi *outlier* (1,5 × IQR)

*Outlier* tanpa penyebab teknis tetap dipertahankan (3.7.4).

In [ ]:
fig, axes = plt.subplots(len(METRICS), 1, figsize=(12, 4 * len(METRICS)))
for ax, (col, (label, unit)) in zip(axes, METRICS.items()):
    groups = [(f'{c}\n{hz}', df.loc[(df.code == c) & (df.hz == hz), col].values) for hz in FREQS for c in CODES]
    ax.boxplot([g for _, g in groups], tick_labels=[n for n, _ in groups])
    ax.set_title(label); ax.set_ylabel(unit)
fig.tight_layout(); fig.savefig(FIGURES / 'boxplot.png', dpi=200)

def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

rows = []
for col in METRICS:
    flag = df.groupby(['code', 'hz'])[col].transform(iqr_outliers).astype(bool)
    for _, r in df[flag].iterrows():
        rows.append({'metrik': col, 'code': r.code, 'hz': r.hz, 'run': r.run, 'nilai': r[col], 'file': r.file})
outliers = pd.DataFrame(rows, columns=['metrik', 'code', 'hz', 'run', 'nilai', 'file'])
outliers.to_csv(TABLES / 'outlier.csv', index=False)
outliers

## 4. Statistik deskriptif (Tabel 3.10)

In [ ]:
def describe(s):
    s = s.dropna(); n = len(s); mean = s.mean(); sd = s.std(ddof=1)
    half = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n) if n > 1 else np.nan
    return pd.Series({'n': n, 'mean': mean, 'median': s.median(), 'sd': sd,
                      'min': s.min(), 'max': s.max(), 'ci95_low': mean - half, 'ci95_high': mean + half})

desc = pd.concat({col: df.groupby(['code', 'hz'])[col].apply(describe).unstack() for col in METRICS},
                 names=['metrik']).round(4)
desc.to_csv(TABLES / 'deskriptif.csv')
desc

## 5. Uji normalitas Shapiro-Wilk per kelompok (16 kelompok × 3 metrik), α = 0,05

Kelompok dengan variansi nol (semua nilai identik) tidak dapat diuji Shapiro-Wilk dan diperlakukan **tidak normal**.

In [ ]:
def shapiro(s):
    s = s.dropna()
    if len(s) < 3 or s.nunique() == 1:
        return pd.Series({'W': np.nan, 'p': np.nan, 'normal': False})
    w, p = stats.shapiro(s)
    return pd.Series({'W': w, 'p': p, 'normal': p > ALPHA})

normality = pd.concat({col: df.groupby(['code', 'hz'])[col].apply(shapiro).unstack() for col in METRICS},
                      names=['metrik'])
normality['normal'] = normality['normal'].astype(bool)
normality.to_csv(TABLES / 'shapiro_wilk.csv')
normality.round(4)

## 6–8. Uji beda, koreksi Holm-Bonferroni, dan *effect size*

24 perbandingan: (A-N vs B-N, A-S vs B-S) × 4 frekuensi × 3 metrik.

- Kedua kelompok normal → Welch's t-test; *effect size* Hedges' g.
- Salah satu tidak normal → Mann-Whitney U (dua sisi); *effect size* *rank-biserial correlation* r.
- Koreksi Holm-Bonferroni atas ke-24 nilai p.

Interpretasi (Cohen, 1988): |g| < 0,2 dapat diabaikan, 0,2 kecil, 0,5 sedang, 0,8 besar;
|r| < 0,1 dapat diabaikan, 0,1 kecil, 0,3 sedang, 0,5 besar.
Tanda positif berarti nilai Provider (A) **lebih besar** daripada Riverpod (B).

In [ ]:
def hedges_g(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    if sp == 0:
        return 0.0 if a.mean() == b.mean() else np.sign(a.mean() - b.mean()) * np.inf
    j = 1 - 3 / (4 * (na + nb) - 9)
    return j * (a.mean() - b.mean()) / sp

def rank_biserial(u_a, na, nb):
    return 2 * u_a / (na * nb) - 1

def interpret(es, kind):
    x = abs(es)
    cuts = (0.2, 0.5, 0.8) if kind == 'g' else (0.1, 0.3, 0.5)
    return ('dapat diabaikan' if x < cuts[0] else 'kecil' if x < cuts[1] else 'sedang' if x < cuts[2] else 'besar')

rows = []
for pattern in ['N', 'S']:
    for hz in FREQS:
        for col in METRICS:
            a = df.loc[(df.code == f'A-{pattern}') & (df.hz == hz), col].dropna()
            b = df.loc[(df.code == f'B-{pattern}') & (df.hz == hz), col].dropna()
            both_normal = bool(normality.loc[(col, f'A-{pattern}', hz), 'normal']
                               and normality.loc[(col, f'B-{pattern}', hz), 'normal'])
            if both_normal:
                stat, p = stats.ttest_ind(a, b, equal_var=False)
                test, es, es_kind = 'Welch t', hedges_g(a, b), 'g'
            else:
                res = stats.mannwhitneyu(a, b, alternative='two-sided')
                stat, p = res.statistic, res.pvalue
                test, es, es_kind = 'Mann-Whitney U', rank_biserial(res.statistic, len(a), len(b)), 'r'
            rows.append({'pola': pattern, 'hz': hz, 'metrik': col,
                         'mean_A': a.mean(), 'mean_B': b.mean(),
                         'selisih_pct': (a.mean() / b.mean() - 1) * 100 if b.mean() else np.nan,
                         'uji': test, 'statistik': stat, 'p': p,
                         'effect_size': es, 'jenis_es': es_kind, 'interpretasi': interpret(es, es_kind)})

comp = pd.DataFrame(rows)
assert len(comp) == 24, len(comp)
p_for_holm = comp['p'].fillna(1.0)
comp['signifikan_holm'], comp['p_holm'] = multipletests(p_for_holm, alpha=ALPHA, method='holm')[:2]
comp.to_csv(TABLES / 'uji_beda_holm.csv', index=False)
comp.round(4)

## 9. Grafik utama: metrik vs frekuensi (rata-rata ± CI 95%)

In [ ]:
styles = {'A-N': ('tab:blue', '-', 'o'), 'A-S': ('tab:blue', '--', 's'),
          'B-N': ('tab:red', '-', 'o'), 'B-S': ('tab:red', '--', 's')}
for col, (label, unit) in METRICS.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for c in CODES:
        d = desc.loc[col].loc[c].reindex(FREQS)
        color, ls, marker = styles[c]
        ax.errorbar(FREQS, d['mean'], yerr=[d['mean'] - d['ci95_low'], d['ci95_high'] - d['mean']],
                    label=c, color=color, linestyle=ls, marker=marker, capsize=4)
    ax.set_xlabel('Frekuensi state update (Hz)'); ax.set_ylabel(f'{label} ({unit})')
    ax.set_xticks(FREQS); ax.grid(alpha=.3); ax.legend(title='Perlakuan')
    ax.set_title(label)
    fig.tight_layout(); fig.savefig(FIGURES / f"grafik_{col.replace('.', '_')}.png", dpi=200)
    plt.show()

## Data pendukung

*Rebuild* `CustomerTile`, persentil *frame time*, *frame* melebihi anggaran, dan RSS maksimum.

In [ ]:
support_cols = ['rebuilds_per_s.CustomerTile', 'frames.total_p90_ms', 'frames.total_p99_ms',
                'frames.build_mean_ms', 'frames.raster_mean_ms', 'frames.count', 'frames.over_budget',
                'memory_mb.max', 'memory_mb.start', 'memory_mb.end']
support_cols = [c for c in support_cols if c in df.columns]
support = df.groupby(['code', 'hz'])[support_cols].mean().round(3)
support.to_csv(TABLES / 'pendukung.csv')
support

## 10. Ekspor

Semua tabel berada di `results/tables/` dan grafik di `results/figures/`.

In [ ]:
for p in sorted(TABLES.glob('*.csv')) + sorted(FIGURES.glob('*.png')):
    print(p.relative_to(OUT_DIR))